In [181]:
import pandas as pd

In [182]:
df_dim_region = pd.read_parquet('../../data/gold/data_warehouse/dw_dim_region.parquet')
df_dim_product = pd.read_parquet('../../data/gold/data_warehouse/dw_dim_product.parquet')
df_dim_supplier = pd.read_parquet('../../data/gold/data_warehouse/dw_dim_supplier.parquet')
df_dim_calendar = pd.read_parquet('../../data/gold/data_warehouse/dw_dim_weekly_calendar.parquet')
df_dim_time_series = pd.read_parquet('../../data/gold/data_warehouse/dw_dim_time_series.parquet')
df_fact_sales = pd.read_parquet('../../data/gold/data_warehouse/dw_fact_sales_weekly.parquet')


In [183]:
df_fact_sales.columns

Index(['week_date', 'supplier_id', 'region_id', 'product_id', 'time_series_id',
       'units_sold'],
      dtype='object')

In [184]:
df_fact_sales.head(5)

,week_date,supplier_id,region_id,product_id,time_series_id,units_sold
0,2022-10-31,1,5,97,3,2
1,2022-10-31,1,5,121,4,61
2,2022-10-31,1,5,134,5,189
3,2022-10-31,1,5,241,12,5064
4,2022-10-31,1,5,246,13,22


In [185]:
df_dim_time_series.head(5)

,time_series_id,supplier_id,region_id,product_id,first_week_date,last_week_date,total_weeks_length,num_week_with_sales,num_week_with_zeros,sales_units,avg_weekly_sales,avg_weekly_sales_non_zero,std_weekly_sales,std_weekly_sales_non_zero,max_weekly_sales,min_weekly_sales,q25_sales,q50_sales,q75_sales,sales_weeks_ratio,cv,iqr
0,1,1,5,69,2023-01-09,2023-08-21,33,4,29,10.0,0.303030,2.500000,1.103541,2.380476,6.0,0.0,0.00,0.0,0.00,0.121212,3.641674,0.0
1,2,1,5,89,2023-03-06,2024-10-21,86,33,53,1716.0,19.953488,52.000000,65.791275,98.889142,375.0,0.0,0.00,0.0,6.00,0.383721,3.297232,6.0
2,3,1,5,97,2022-10-31,2023-01-30,14,3,11,5.0,0.357143,1.666667,0.744946,0.577350,2.0,0.0,0.00,0.0,0.00,0.214286,2.085844,0.0
3,4,1,5,121,2022-10-31,2024-10-21,104,83,21,9966.0,95.826923,120.072289,90.341149,85.469119,393.0,0.0,17.25,82.0,145.25,0.798077,0.942753,128.0
4,5,1,5,134,2022-10-31,2024-10-21,104,48,56,14208.0,136.615385,296.000000,670.247126,967.627918,5646.0,0.0,0.00,0.0,39.00,0.461538,4.906088,39.0


In [186]:
df_dim_time_series

,time_series_id,supplier_id,region_id,product_id,first_week_date,last_week_date,total_weeks_length,num_week_with_sales,num_week_with_zeros,sales_units,avg_weekly_sales,avg_weekly_sales_non_zero,std_weekly_sales,std_weekly_sales_non_zero,max_weekly_sales,min_weekly_sales,q25_sales,q50_sales,q75_sales,sales_weeks_ratio,cv,iqr
0,1,1,5,69,2023-01-09,2023-08-21,33,4,29,10.0,0.303030,2.500000,1.103541,2.380476,6.0,0.0,0.00,0.0,0.00,0.121212,3.641674,0.0
1,2,1,5,89,2023-03-06,2024-10-21,86,33,53,1716.0,19.953488,52.000000,65.791275,98.889142,375.0,0.0,0.00,0.0,6.00,0.383721,3.297232,6.0
2,3,1,5,97,2022-10-31,2023-01-30,14,3,11,5.0,0.357143,1.666667,0.744946,0.577350,2.0,0.0,0.00,0.0,0.00,0.214286,2.085844,0.0
3,4,1,5,121,2022-10-31,2024-10-21,104,83,21,9966.0,95.826923,120.072289,90.341149,85.469119,393.0,0.0,17.25,82.0,145.25,0.798077,0.942753,128.0
4,5,1,5,134,2022-10-31,2024-10-21,104,48,56,14208.0,136.615385,296.000000,670.247126,967.627918,5646.0,0.0,0.00,0.0,39.00,0.461538,4.906088,39.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2836,2841,27,4,351,2022-11-28,2023-11-20,52,5,47,17.0,0.326923,3.400000,1.688730,4.827007,12.0,0.0,0.00,0.0,0.00,0.096154,5.165512,0.0
2837,2842,27,4,357,2022-11-07,2024-10-07,101,26,75,58.0,0.574257,2.230769,1.321715,1.773306,7.0,0.0,0.00,0.0,1.00,0.257426,2.301603,1.0
2838,2843,27,4,373,2022-12-05,2024-10-21,99,44,55,178.0,1.797980,4.045455,3.401620,4.131302,19.0,0.0,0.00,0.0,2.00,0.444444,1.891911,2.0
2839,2844,27,4,377,2022-10-31,2024-09-30,101,67,34,263.0,2.603960,3.925373,2.946453,2.808609,13.0,0.0,0.00,2.0,4.00,0.663366,1.131527,4.0


In [187]:
rows = []
for _, r in df_dim_time_series.iterrows():
    ts_id = r['time_series_id']
    start = pd.to_datetime(r['first_week_date'])
    end = pd.to_datetime(r['last_week_date'])
    if pd.isna(start) or pd.isna(end):
        continue
    for d in pd.date_range(start=start, end=end, freq='W-MON'):
        rows.append({'time_series_id': ts_id, 'week_date': d})
# Create a new DataFrame from the list of rows
df_spines_time_series = pd.DataFrame(rows)

In [188]:
pd.set_option('display.max_columns', None)

In [189]:
def generate_spines_time_series(df_fact_sales, df_dim_time_series):
    """
    Generate a DataFrame with one row per time series and week date, covering the entire date range for each time series.
    This is done by iterating over each time series in df_dim_time_series, creating a date range from the first_week_date 
    to the last_week_date, and then merging this with df_fact_sales to get the sales data for each week.
    """
    # Create a list to hold the rows of the new DataFrame
    rows = []
    for _, r in df_dim_time_series.iterrows():
        ts_id = r['time_series_id']
        start = pd.to_datetime(r['first_week_date'])
        end = pd.to_datetime(r['last_week_date'])
        if pd.isna(start) or pd.isna(end):
            continue
        for d in pd.date_range(start=start, end=end, freq='W-MON'):
            rows.append({'time_series_id': ts_id, 'week_date': d})
    # Create a new DataFrame from the list of rows
    df_spines_time_series = pd.DataFrame(rows)

    # Convert the 'week_date' column to datetime and sort the DataFrame
    if not df_spines_time_series.empty:
        df_spines_time_series['week_date'] = pd.to_datetime(df_spines_time_series['week_date'])
        df_spines_time_series = df_spines_time_series.sort_values(['time_series_id', 'week_date']).reset_index(drop=True)
    
    # Merge the spines time series with the fact sales data to get the sales for each week
    
    df_spine_sales = (df_spines_time_series
            .merge(df_dim_time_series, on='time_series_id', how='left')
            .merge(df_fact_sales, on=['time_series_id', 'supplier_id','region_id','product_id','week_date'], how='left')
            .fillna(0)
            )
    
    return df_spine_sales

In [190]:
df = generate_spines_time_series(df_fact_sales,df_dim_time_series)
df

,time_series_id,week_date,supplier_id,region_id,product_id,first_week_date,last_week_date,total_weeks_length,num_week_with_sales,num_week_with_zeros,sales_units,avg_weekly_sales,avg_weekly_sales_non_zero,std_weekly_sales,std_weekly_sales_non_zero,max_weekly_sales,min_weekly_sales,q25_sales,q50_sales,q75_sales,sales_weeks_ratio,cv,iqr,units_sold
0,1,2023-01-09,1,5,69,2023-01-09,2023-08-21,33,4,29,10.0,0.303030,2.500000,1.103541,2.380476,6.0,0.0,0.0,0.0,0.0,0.121212,3.641674,0.0,2.0
1,1,2023-01-16,1,5,69,2023-01-09,2023-08-21,33,4,29,10.0,0.303030,2.500000,1.103541,2.380476,6.0,0.0,0.0,0.0,0.0,0.121212,3.641674,0.0,6.0
2,1,2023-01-23,1,5,69,2023-01-09,2023-08-21,33,4,29,10.0,0.303030,2.500000,1.103541,2.380476,6.0,0.0,0.0,0.0,0.0,0.121212,3.641674,0.0,0.0
3,1,2023-01-30,1,5,69,2023-01-09,2023-08-21,33,4,29,10.0,0.303030,2.500000,1.103541,2.380476,6.0,0.0,0.0,0.0,0.0,0.121212,3.641674,0.0,0.0
4,1,2023-02-06,1,5,69,2023-01-09,2023-08-21,33,4,29,10.0,0.303030,2.500000,1.103541,2.380476,6.0,0.0,0.0,0.0,0.0,0.121212,3.641674,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203466,2845,2024-07-29,27,4,381,2024-04-29,2024-08-26,18,3,15,11.0,0.611111,3.666667,1.577000,2.081666,6.0,0.0,0.0,0.0,0.0,0.166667,2.580541,0.0,0.0
203467,2845,2024-08-05,27,4,381,2024-04-29,2024-08-26,18,3,15,11.0,0.611111,3.666667,1.577000,2.081666,6.0,0.0,0.0,0.0,0.0,0.166667,2.580541,0.0,0.0
203468,2845,2024-08-12,27,4,381,2024-04-29,2024-08-26,18,3,15,11.0,0.611111,3.666667,1.577000,2.081666,6.0,0.0,0.0,0.0,0.0,0.166667,2.580541,0.0,0.0
203469,2845,2024-08-19,27,4,381,2024-04-29,2024-08-26,18,3,15,11.0,0.611111,3.666667,1.577000,2.081666,6.0,0.0,0.0,0.0,0.0,0.166667,2.580541,0.0,0.0


In [191]:
df['sales_units'] = df['sales_units'].astype(int)
df['max_weekly_sales'] = df['max_weekly_sales'].astype(int)
df['min_weekly_sales'] = df['min_weekly_sales'].astype(int)
df['q25_sales'] = df['q25_sales'].astype(int)
df['q50_sales'] = df['q50_sales'].astype(int)
df['q75_sales'] = df['q75_sales'].astype(int)
df['units_sold'] = df['units_sold'].astype(int)

In [192]:
def bring_dimensions(df, df_dim_region, df_dim_product, df_dim_supplier,df_dim_calendar):
    """
    Merge the fact sales data with the dimension tables to bring in the relevant attributes for each time series.
    This is done by merging the fact sales DataFrame with each of the dimension DataFrames on the appropriate keys.
    """
    df = (df
          .merge(df_dim_region, on='region_id', how='left')
          .merge(df_dim_product, on='product_id', how='left')
          .merge(df_dim_supplier, on='supplier_id', how='left')
          .merge(df_dim_calendar, on='week_date', how='left')
         )
    return df

In [193]:
df = bring_dimensions(df, df_dim_region, df_dim_product, df_dim_supplier, df_dim_calendar)
df.head(5)

,time_series_id,week_date,supplier_id,region_id,product_id,first_week_date,last_week_date,total_weeks_length,num_week_with_sales,num_week_with_zeros,sales_units,avg_weekly_sales,avg_weekly_sales_non_zero,std_weekly_sales,std_weekly_sales_non_zero,max_weekly_sales,min_weekly_sales,q25_sales,q50_sales,q75_sales,sales_weeks_ratio,cv,iqr,units_sold,region_name,product_attribute_1,product_attribute_2,product_attribute_3,product_name,supplier_name,week,start_date,end_date,year,semester,semester_date,semester_name,quarter,quarter_date,quarter_name,month,month_name,month_date
0,1,2023-01-09,1,5,69,2023-01-09,2023-08-21,33,4,29,10,0.30303,2.5,1.103541,2.380476,6,0,0,0,0,0.121212,3.641674,0.0,2,Sul,A,A,A04,Product 69,Supplier 1,2,2023-01-09,2023-01-15,2023,1,2023-01-01,H1,1,2023-01-01,Q1,1,January,2023-01-01
1,1,2023-01-16,1,5,69,2023-01-09,2023-08-21,33,4,29,10,0.30303,2.5,1.103541,2.380476,6,0,0,0,0,0.121212,3.641674,0.0,6,Sul,A,A,A04,Product 69,Supplier 1,3,2023-01-16,2023-01-22,2023,1,2023-01-01,H1,1,2023-01-01,Q1,1,January,2023-01-01
2,1,2023-01-23,1,5,69,2023-01-09,2023-08-21,33,4,29,10,0.30303,2.5,1.103541,2.380476,6,0,0,0,0,0.121212,3.641674,0.0,0,Sul,A,A,A04,Product 69,Supplier 1,4,2023-01-23,2023-01-29,2023,1,2023-01-01,H1,1,2023-01-01,Q1,1,January,2023-01-01
3,1,2023-01-30,1,5,69,2023-01-09,2023-08-21,33,4,29,10,0.30303,2.5,1.103541,2.380476,6,0,0,0,0,0.121212,3.641674,0.0,0,Sul,A,A,A04,Product 69,Supplier 1,5,2023-01-30,2023-02-05,2023,1,2023-01-01,H1,1,2023-01-01,Q1,1,January,2023-01-01
4,1,2023-02-06,1,5,69,2023-01-09,2023-08-21,33,4,29,10,0.30303,2.5,1.103541,2.380476,6,0,0,0,0,0.121212,3.641674,0.0,0,Sul,A,A,A04,Product 69,Supplier 1,6,2023-02-06,2023-02-12,2023,1,2023-02-01,H1,1,2023-01-01,Q1,2,February,2023-02-01


In [194]:
df.shape

(203471, 43)

In [195]:
df.columns

Index(['time_series_id', 'week_date', 'supplier_id', 'region_id', 'product_id',
       'first_week_date', 'last_week_date', 'total_weeks_length',
       'num_week_with_sales', 'num_week_with_zeros', 'sales_units',
       'avg_weekly_sales', 'avg_weekly_sales_non_zero', 'std_weekly_sales',
       'std_weekly_sales_non_zero', 'max_weekly_sales', 'min_weekly_sales',
       'q25_sales', 'q50_sales', 'q75_sales', 'sales_weeks_ratio', 'cv', 'iqr',
       'units_sold', 'region_name', 'product_attribute_1',
       'product_attribute_2', 'product_attribute_3', 'product_name',
       'supplier_name', 'week', 'start_date', 'end_date', 'year', 'semester',
       'semester_date', 'semester_name', 'quarter', 'quarter_date',
       'quarter_name', 'month', 'month_name', 'month_date'],
      dtype='object')

In [196]:
import pandas as pd
import numpy as np

# =========================================================================
# ORGANIZAR E OTIMIZAR BASE_DATASET
# =========================================================================

def optimize_base_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Organiza colunas e otimiza tipos de dados do base_dataset
    
    Parameters:
    -----------
    df : pd.DataFrame
        Base dataset completo
    
    Returns:
    --------
    pd.DataFrame : Dataset otimizado
    """
    
    print("🔧 Otimizando base_dataset...")
    print(f"   Shape inicial: {df.shape}")
    print(f"   Memória inicial: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    df = df.copy()
    
    # =========================================================================
    # 1. ORGANIZAR COLUNAS EM ORDEM LÓGICA
    # =========================================================================
    
    primary_keys = ['time_series_id', 'week_date']
    
    dimension_keys = ['supplier_id', 'region_id', 'product_id']
    
    target = ['units_sold']
    
    # Calendar features from df_dim_calendar (joined on week_date)
    calendar_features = [
        'week',
        'start_date',
        'end_date',
        'year',
        'semester',
        'semester_date',
        'semester_name',
        'quarter',
        'quarter_date',
        'quarter_name',
        'month',
        'month_name',
        'month_date',
    ]
    
    temporal_metadata = ['first_week_date', 'last_week_date', 'total_weeks_length']
    
    statistical_metadata = [
        'num_week_with_sales',
        'num_week_with_zeros',
        'sales_weeks_ratio',
        'sales_units',
        'avg_weekly_sales',
        'avg_weekly_sales_non_zero',
        'std_weekly_sales',
        'std_weekly_sales_non_zero',
        'max_weekly_sales',
        'min_weekly_sales',
        'q25_sales',
        'q50_sales',
        'q75_sales',
        'iqr',
        'cv',
    ]
    
    dimension_attributes = [
        'supplier_name',
        'region_name',
        'product_name',
        'product_attribute_1',
        'product_attribute_2',
        'product_attribute_3',
    ]
    
    column_order = (
        primary_keys +
        dimension_keys +
        target +
        calendar_features +
        temporal_metadata +
        statistical_metadata +
        dimension_attributes
    )
    
    missing_cols = [col for col in column_order if col not in df.columns]
    if missing_cols:
        print(f"   ⚠️  Colunas ausentes (verifique bring_dimensions): {missing_cols}")
    
    existing_order = [col for col in column_order if col in df.columns]
    
    # Preserve any unexpected extra columns at the end
    remaining_cols = [col for col in df.columns if col not in existing_order]
    if remaining_cols:
        print(f"   ℹ️  Colunas adicionais preservadas: {remaining_cols}")
        existing_order.extend(remaining_cols)
    
    df = df[existing_order]
    print(f"   ✅ Colunas reordenadas ({len(existing_order)} total)")
    
    # =========================================================================
    # 2. OTIMIZAR TIPOS DE DADOS
    # =========================================================================
    
    print("\n📊 Otimizando tipos de dados...")
    
    # IDs
    if 'time_series_id' in df.columns:
        df['time_series_id'] = df['time_series_id'].astype(
            'int32' if df['time_series_id'].nunique() < 50_000 else 'int64'
        )
    for col in ['supplier_id', 'region_id', 'product_id']:
        if col in df.columns:
            df[col] = df[col].astype('category')
    print("   ✅ IDs otimizados")
    
    # Datas
    date_cols = [
        'week_date', 'first_week_date', 'last_week_date',
        'start_date', 'end_date',          # from dim_calendar
        'semester_date', 'quarter_date', 'month_date',
    ]
    for col in date_cols:
        if col in df.columns and not pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = pd.to_datetime(df[col])
    print("   ✅ Datas em datetime64")
    
    # Inteiros temporais (year, semester, quarter, month, week)
    for col in ['year', 'semester', 'quarter', 'month', 'week']:
        if col in df.columns:
            df[col] = df[col].astype('int16')
    
    # Nomes categóricos do calendário
    for col in ['semester_name', 'quarter_name', 'month_name']:
        if col in df.columns:
            df[col] = df[col].astype('category')
    print("   ✅ Features de calendário otimizadas")
    
    # Contagens → Int64
    count_columns = [
        'units_sold', 'sales_units',
        'max_weekly_sales', 'min_weekly_sales'
        'q25_sales', 'q50_sales', 'q75_sales',
        'num_week_with_sales', 'num_week_with_zeros','total_weeks_length',
    ]
    for col in count_columns:
        if col in df.columns:
            df[col] = df[col].astype('Int64')
    print(f"   ✅ Contagens → Int64 ({sum(c in df.columns for c in count_columns)} colunas)")
    
    # Contínuas → float32
    continuous_columns = [
        'avg_weekly_sales', 'std_weekly_sales', 'cv', 'iqr', 'sales_weeks_ratio',
    ]
    for col in continuous_columns:
        if col in df.columns:
            df[col] = df[col].astype('float32')
    print(f"   ✅ Métricas contínuas → float32 ({sum(c in df.columns for c in continuous_columns)} colunas)")
    
    # Categóricas dimensionais
    categorical_columns = [
        'supplier_name', 'region_name', 'product_name',
        'product_attribute_1', 'product_attribute_2', 'product_attribute_3',
    ]
    for col in categorical_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    print(f"   ✅ Categóricas → category ({sum(c in df.columns for c in categorical_columns)} colunas)")
    
    # =========================================================================
    # 3. RELATÓRIO FINAL
    # =========================================================================
    
    print("\n" + "="*70)
    print("✅ BASE_DATASET OTIMIZADO!")
    print("="*70)
    print(f"\n📊 Shape: {df.shape}")
    print(f"   Memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"\n📋 Tipos:\n{df.dtypes.value_counts()}")
    
    display_cols = ['time_series_id', 'week_date', 'units_sold', 'week', 'year', 'month', 'quarter', 'avg_weekly_sales']
    display_cols = [c for c in display_cols if c in df.columns]
    print(f"\n🔍 Amostra:\n{df[display_cols].head(3).to_string(index=False)}")
    
    return df


# =========================================================================
# USO
# =========================================================================

df_base_optimized = optimize_base_dataset(df)

df_base_optimized.to_parquet('../../data/gold/forecasting/ds_base_dataset.parquet')
print("\n💾 Salvo em data/gold/forecasting/ds_base_dataset.parquet")

🔧 Otimizando base_dataset...
   Shape inicial: (203471, 43)
   Memória inicial: 150.79 MB
   ✅ Colunas reordenadas (43 total)

📊 Otimizando tipos de dados...
   ✅ IDs otimizados
   ✅ Datas em datetime64
   ✅ Features de calendário otimizadas
   ✅ Contagens → Int64 (8 colunas)
   ✅ Métricas contínuas → float32 (5 colunas)
   ✅ Categóricas → category (6 colunas)

✅ BASE_DATASET OTIMIZADO!

📊 Shape: (203471, 43)
   Memória: 40.42 MB

📋 Tipos:
Int64             8
datetime64[ns]    8
float32           5
int16             5
int32             3
category          2
float64           2
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
category          1
Name: count, dtype: int64

🔍 Amostra:
 time_series_id  week_date  units_sold  week  year  month  quarter  avg_weekly_sales
              1 2023-01-09           2     2  2023      1        1           0.30303
        

In [197]:
df_base_optimized

,time_series_id,week_date,supplier_id,region_id,product_id,units_sold,week,start_date,end_date,year,semester,semester_date,semester_name,quarter,quarter_date,quarter_name,month,month_name,month_date,first_week_date,last_week_date,total_weeks_length,num_week_with_sales,num_week_with_zeros,sales_weeks_ratio,sales_units,avg_weekly_sales,avg_weekly_sales_non_zero,std_weekly_sales,std_weekly_sales_non_zero,max_weekly_sales,min_weekly_sales,q25_sales,q50_sales,q75_sales,iqr,cv,supplier_name,region_name,product_name,product_attribute_1,product_attribute_2,product_attribute_3
0,1,2023-01-09,1,5,69,2,2,2023-01-09,2023-01-15,2023,1,2023-01-01,H1,1,2023-01-01,Q1,1,January,2023-01-01,2023-01-09,2023-08-21,33,4,29,0.121212,10,0.303030,2.500000,1.103541,2.380476,6,0,0,0,0,0.0,3.641674,Supplier 1,Sul,Product 69,A,A,A04
1,1,2023-01-16,1,5,69,6,3,2023-01-16,2023-01-22,2023,1,2023-01-01,H1,1,2023-01-01,Q1,1,January,2023-01-01,2023-01-09,2023-08-21,33,4,29,0.121212,10,0.303030,2.500000,1.103541,2.380476,6,0,0,0,0,0.0,3.641674,Supplier 1,Sul,Product 69,A,A,A04
2,1,2023-01-23,1,5,69,0,4,2023-01-23,2023-01-29,2023,1,2023-01-01,H1,1,2023-01-01,Q1,1,January,2023-01-01,2023-01-09,2023-08-21,33,4,29,0.121212,10,0.303030,2.500000,1.103541,2.380476,6,0,0,0,0,0.0,3.641674,Supplier 1,Sul,Product 69,A,A,A04
3,1,2023-01-30,1,5,69,0,5,2023-01-30,2023-02-05,2023,1,2023-01-01,H1,1,2023-01-01,Q1,1,January,2023-01-01,2023-01-09,2023-08-21,33,4,29,0.121212,10,0.303030,2.500000,1.103541,2.380476,6,0,0,0,0,0.0,3.641674,Supplier 1,Sul,Product 69,A,A,A04
4,1,2023-02-06,1,5,69,0,6,2023-02-06,2023-02-12,2023,1,2023-02-01,H1,1,2023-01-01,Q1,2,February,2023-02-01,2023-01-09,2023-08-21,33,4,29,0.121212,10,0.303030,2.500000,1.103541,2.380476,6,0,0,0,0,0.0,3.641674,Supplier 1,Sul,Product 69,A,A,A04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203466,2845,2024-07-29,27,4,381,0,31,2024-07-29,2024-08-04,2024,2,2024-07-01,H2,3,2024-07-01,Q3,7,July,2024-07-01,2024-04-29,2024-08-26,18,3,15,0.166667,11,0.611111,3.666667,1.577000,2.081666,6,0,0,0,0,0.0,2.580541,Supplier 27,Sudeste,Product 381,B,C,B31
203467,2845,2024-08-05,27,4,381,0,32,2024-08-05,2024-08-11,2024,2,2024-08-01,H2,3,2024-07-01,Q3,8,August,2024-08-01,2024-04-29,2024-08-26,18,3,15,0.166667,11,0.611111,3.666667,1.577000,2.081666,6,0,0,0,0,0.0,2.580541,Supplier 27,Sudeste,Product 381,B,C,B31
203468,2845,2024-08-12,27,4,381,0,33,2024-08-12,2024-08-18,2024,2,2024-08-01,H2,3,2024-07-01,Q3,8,August,2024-08-01,2024-04-29,2024-08-26,18,3,15,0.166667,11,0.611111,3.666667,1.577000,2.081666,6,0,0,0,0,0.0,2.580541,Supplier 27,Sudeste,Product 381,B,C,B31
203469,2845,2024-08-19,27,4,381,0,34,2024-08-19,2024-08-25,2024,2,2024-08-01,H2,3,2024-07-01,Q3,8,August,2024-08-01,2024-04-29,2024-08-26,18,3,15,0.166667,11,0.611111,3.666667,1.577000,2.081666,6,0,0,0,0,0.0,2.580541,Supplier 27,Sudeste,Product 381,B,C,B31
